# WeatherScope AI — Pipeline Walkthrough

Runs against the artifacts produced by `python main.py --stage all`.
Each section loads a pipeline output and shows how to explore it interactively.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from src.utils import load_config, resolve_path

config = load_config()
clean = pd.read_parquet(resolve_path(config['paths']['processed_data']))
clean.head()

## Cleaned dataset overview

In [ ]:
clean.describe().T.head(15)

## Daily forecasting series

In [ ]:
daily = pd.read_parquet(resolve_path('data/processed/daily_series.parquet'))
daily.plot(subplots=True, figsize=(12, 8));

## Model comparison (holdout metrics)

In [ ]:
import json
metrics = json.loads(resolve_path('outputs/reports/forecast_metrics.json').read_text())
pd.DataFrame(metrics['temperature_celsius']).T.sort_values('RMSE').round(3)

## Actual vs predicted on the holdout window

In [ ]:
preds = pd.read_parquet(resolve_path('outputs/models/forecast_predictions.parquet'))
temp = preds[(preds.target == 'temperature_celsius') & preds.model.isin(['ARIMA', 'XGBoost'])]
ax = temp.drop_duplicates('date').plot(x='date', y='y_true', color='gray', figsize=(12, 5), label='actual')
for name, group in temp.groupby('model'):
    group.plot(x='date', y='y_pred', ax=ax, label=name)